# LFRic Iris data manipulation and visualisation practical

Let's apply what we've learned in Section 01 to Section 05 about data processing and visualisation of LFRic data in Iris and PyVista with the two exercises. The aim is to use the prompts to write the code yourself, but we have also provided a [separate notebook](./answers/06_Exercise_01_answers.ipynb) containing the answers if you are stuck. All the information needed to write the code for this practical can be found in the notebooks in the first part of this practical. 

Note: This is delivered in JupyterLabs, but sometime the PyVista and GeoVista plotting is laggy in labs. If you prefer you can run in IPython.

## Step 0: Conda setup (required for regridding)

Run these commands in a **fresh terminal** before running notebook cells:

```bash
cd <path to directory>/LFRic-Atmosphere-Training
conda create -n lfric-mesh python=3.12 -y
conda activate lfric-mesh
conda install -c conda-forge esmpy -y
# zsh-safe quoting for extras is important
python -m pip install -e '.[mesh_tutorials]'
python -m pip install jupyterlab
python -m pip show esmf-regrid
python -c "import esmf_regrid, ESMF; print('Imports OK')"
python -m jupyter lab --version
python -m ipykernel install --user --name lfric-mesh --display-name "Python (lfric-mesh)"
python -m jupyter lab
```

If `python -m pip show esmf-regrid` returns nothing, the package is not installed in that env.
If `import esmf_regrid` fails, rerun the quoted install command exactly as shown.

Then in Jupyter choose **Kernel -> Change Kernel -> Python (lfric-mesh)** and restart the kernel.

Reason: `esmf-regrid` needs the compiled ESMF backend (`esmpy`), which is installed from conda-forge.


In [ ]:
# Notebook warning controls
import warnings

# Suppress repeated NumPy masked-array casting warnings from sample data loading.
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in cast",
    category=RuntimeWarning,
)
# Suppress Iris legacy date-precision future warning in tutorial output.
warnings.filterwarnings(
    "ignore",
    message="You are using legacy date precision for Iris units*",
    category=FutureWarning,
)

# Use microsecond date precision to align with upcoming Iris default behaviour.
try:
    import iris
    iris.FUTURE.date_microseconds = True
except Exception:
    pass

# Import cartopy before importing ESMF to avoid a segfault bug.
import cartopy

# Check that this kernel can run regridding
import importlib.util
import sys
print("Kernel Python:", sys.executable)
assert importlib.util.find_spec("ESMF") or importlib.util.find_spec("esmpy"), "ESMPy backend missing in this kernel"
assert importlib.util.find_spec("esmf_regrid"), "esmf_regrid missing in this kernel"
print("Backend check: OK")


## Exercise 1 - Regrid LFRic to UM and plot data 

In this exercise you will take LFRic data, regrid it to UM data and then plot the differences

**Step 1** To begin, we need to import the neccesary packages that we will need for this exercise.

In [ ]:
%matplotlib inline
import pyvista as pv
import geovista as gv
import geovista.theme
import iris.quickplot as qplt
import iris
from geovista import GeoPlotter
from esmf_regrid import ESMFAreaWeighted
pv.set_jupyter_backend("static")
iris.FUTURE.datum_support = True  # avoids some warnings

----
NOTE: the [`iris.experimental.geovista.cube_to_polydata`](https://scitools-iris.readthedocs.io/en/v3.15.0/generated/api/iris.experimental.geovista.html#iris.experimental.geovista.cube_to_polydata) function is provided to convert a cube to a pyvista `polydata` object.  We will be using that ...

In [ ]:
from iris.experimental.geovista import cube_to_polydata

In [ ]:
# Define the location of the data and file names
data_path = '../example_data/'
lfric_path = data_path + 'u-ct674_20210324T0000Z_lf_ugrid.nc'
um_path = data_path + 'u-ct674_20210324T0000Z_um_latlon.nc'

# Continue here with importing the data... 

**Step 2**  
Use iris.load_cube to load in the LFRic data, and select the diagnostic `"surface_air_pressure"`.  
*Print* the cube -- is it a mesh or grid, and how do you know ?

(hint:  use `iris.load_cube(path, "data_name")`  -- see [Strict loading](https://scitools-iris.readthedocs.io/en/v3.15.0/user_manual/tutorial/loading_iris_cubes.html#strict-loading) )

In [ ]:
# Code here...

**Step 3** Extract the first timestep from the data so we have a 2D cube  
( hint: index cubes with [] -- see [Cube indexing](https://scitools-iris.readthedocs.io/en/v3.15.0/user_manual/tutorial/subsetting_a_cube.html#cube-indexing) )

In [ ]:
# Code here...

**Step 4** Transform the LFRic cube into a pyvista object using the Iris `cube_to_polydata` function

(Note: [`Geovista.bridge.Transform`](https://geovista.readthedocs.io/en/latest/reference/generated/api/geovista/bridge/#geovista.bridge.Transform) does most of the work. See the code [here](https://scitools-iris.readthedocs.io/en/v3.15.0/_modules/iris/experimental/geovista.html#cube_to_polydata))

In [ ]:
# Code here...

**Step 5**  
Use GeoPlotter to plot the data using PyVista.  
You can now observe Surface Air Pressure on a 3D globe.  

(hints:  
create a `GeoPlotter()` ;  
use its `.add_mesh(polydata)` ;  
finally use it's `.show()`  
  -- see [Geovista basic example](https://geovista.readthedocs.io/en/latest/quick_start.html#oisst-avhrr) (but you don't need a color-bar)  
)

In [ ]:
# Code here...

**( Step "5(a)" )**  
In fact, the simplest way of plotting is just to use `polydata.plot()`.  
Try that.

----
Now, lets regrid some LFRic data onto a lat/lon grid

**Step 6**
Use iris.load_cube to load the reference grid and print the cube.  
You can use the equivalent UM data loaded above for this.  

(hints:  
load from `um_path`;  
sub-index as required to get a 2D "grid" cube  
)

In [ ]:
um_cube = iris.load_cube(um_path, "surface_air_pressure")
print(um_cube)

In [ ]:
# Code here...

**Step 7**  
Create an Iris regridding scheme using `ESMFAreaWeighted`.  
Consult the iris-esmf-regrid documentation [here](https://iris-esmf-regrid.readthedocs.io/en/latest/_api_generated/esmf_regrid.schemes.html#esmf_regrid.schemes.ESMFAreaWeighted)

(hint: `ESMFAreaWeighted` doesn't _require_ any arguments)

In [ ]:
# Code here...

**Step 8**  
Now, regrid the LFRic data using the Iris scheme you created.  
*Print* the result -- is your LFRic data regridded?

(hint: use `cube.regrid(grid_cube, scheme)` )

In [ ]:
# Code here...

**Step 9** Use `qplt.pcolormesh` to plot the regridded LFRic data

In [ ]:
# Code here...

----
We can use the UM data loaded as reference for the regridding to compare to the reggrided LFRic data.  

**Step 10**  Select the first timestep of the UM data loaded in step 6

In [ ]:
# Code here...

**Step 11**  Calculate the difference between the LFRic regridded data and native UM data  

(hint: cubes can be simply subtracted -- see [Cube Maths](https://scitools-iris.readthedocs.io/en/v3.15.0/user_manual/tutorial/cube_maths.html) )

In [ ]:
# Code here...

**Step 12** Plot the original UM data and the regridded LFRic data and the difference side by side.  

(hint: use `plt.subplot(1,3,1)`, and try adding a title and coastlines to the plot.)

In [ ]:
# Code here...